In [14]:
import numpy as np
import pandas as pd
import glob
import os


# Data Cleaning

## Overview

This notebook builds the cleaned analytical datasets used throughout the rest of the project. The raw source is the City of Chicago public crime data, restricted to complete yearly files from **2016 through 2025**. The objective is not only to combine the files, but also to standardize field types, address the small amount of missingness in key variables, and create two downstream-ready datasets:

- `df`: the main cleaned dataset used for EDA and modeling
- `df_map`: a map-ready subset that additionally requires valid latitude and longitude

The overall conclusion is straightforward: the raw files are already in relatively good shape, and the remaining cleaning work is focused on targeted fixes rather than large-scale repair. After cleaning, the resulting datasets are structurally consistent and suitable for exploratory analysis, spatial visualization, and predictive modeling.


## Raw Data Assembly

The first step is to read the yearly CSV files, keep only the fields needed for arrest analysis, and concatenate them into one combined table. This establishes the full study sample before any missing-value handling or feature engineering is applied.


In [15]:
folder_path = "/Users/fuyanting/Documents/capstone/data"
files = glob.glob(os.path.join(folder_path, "Crimes_*.csv"))


use_cols = [
    'ID',
    'Date',
    'Primary Type',
    'Description',
    'Location Description',
    'Arrest',
    'Domestic',
    'District',
    'Community Area',
    'Year',
    'Latitude',
    'Longitude'
]

df_list = []

for file in files:
    
    temp = pd.read_csv(
        file,
        usecols=use_cols,
        low_memory=False
    )
    
    df_list.append(temp)


df = pd.concat(df_list, ignore_index=True)
print("\nFinal shape:", df.shape)
print("\nYear distribution:\n", df['Year'].value_counts().sort_index())


Final shape: (2491875, 12)

Year distribution:
 Year
2016    269968
2017    269292
2018    269155
2019    261709
2020    212720
2021    209679
2022    240034
2023    263311
2024    259073
2025    236934
Name: count, dtype: int64


The combined raw dataset contains one row per reported incident and preserves the original timestamp, offense, setting, arrest outcome, domestic indicator, district, community area, year, and coordinates. The preview below is included only as a quick field-level reference.


In [16]:
df.head()

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,District,Community Area,Year,Latitude,Longitude
0,13211439,01/01/2016 12:00:00 AM,OFFENSE INVOLVING CHILDREN,AGGRAVATED SEXUAL ASSAULT OF CHILD BY FAMILY M...,RESIDENCE,False,True,8.0,63.0,2016,NaN,NaN
1,13047536,01/01/2016 12:00:00 AM,OFFENSE INVOLVING CHILDREN,AGGRAVATED SEXUAL ASSAULT OF CHILD BY FAMILY M...,APARTMENT,True,True,10.0,30.0,2016,41.837918,-87.713393
2,12328317,01/01/2016 12:00:00 AM,SEX OFFENSE,CRIMINAL SEXUAL ABUSE,APARTMENT,False,False,10.0,30.0,2016,41.849026,-87.708830
3,12389557,01/01/2016 12:00:00 AM,SEX OFFENSE,SEXUAL EXPLOITATION OF A CHILD,RESIDENCE,True,True,4.0,43.0,2016,NaN,NaN
4,13278509,01/01/2016 12:00:00 AM,SEX OFFENSE,AGGRAVATED CRIMINAL SEXUAL ABUSE,RESIDENCE,False,True,8.0,58.0,2016,NaN,NaN


To understand the starting point for cleaning, the notebook next checks the raw schema and missing-value profile. These outputs answer two practical questions: whether the columns arrive in the expected data types, and whether any missingness affects variables that will later be used in EDA or modeling.


In [17]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2491875 entries, 0 to 2491874
Data columns (total 12 columns):
 #   Column                Dtype  
---  ------                -----  
 0   ID                    int64  
 1   Date                  object 
 2   Primary Type          object 
 3   Description           object 
 4   Location Description  object 
 5   Arrest                bool   
 6   Domestic              bool   
 7   District              float64
 8   Community Area        float64
 9   Year                  int64  
 10  Latitude              float64
 11  Longitude             float64
dtypes: bool(2), float64(4), int64(2), object(4)
memory usage: 194.9+ MB


In [18]:
print(df.isna().sum())
print('-----------')
print((df.isna().mean() * 100).round(2))

ID                          0
Date                        0
Primary Type                0
Description                 0
Location Description    13168
Arrest                      0
Domestic                    0
District                    1
Community Area            206
Year                        0
Latitude                36957
Longitude               36957
dtype: int64
-----------
ID                      0.00
Date                    0.00
Primary Type            0.00
Description             0.00
Location Description    0.53
Arrest                  0.00
Domestic                0.00
District                0.00
Community Area          0.01
Year                    0.00
Latitude                1.48
Longitude               1.48
dtype: float64


## Cleaning Rules and Transformations

The cleaning strategy is intentionally conservative and closely tied to the project question. At this stage, the notebook focuses first on the raw missing-value pattern and applies only the row-level rules needed to define the main cleaned datasets:

- Keep the core variables required for arrest prediction and descriptive analysis
- Fill missing `Location Description` values with `Unknown` rather than dropping those incidents
- Drop rows missing `District` or `Community Area`, because those are essential spatial fields for later analysis
- Create `df_map` as a stricter subset that also requires valid `Latitude` and `Longitude`

These choices preserve as much information as possible while defining the two cleaned datasets before any downstream feature engineering is introduced.


In [19]:
df['Location Description'] = df['Location Description'].fillna('Unknown')
df = df.dropna(subset=['Community Area', 'District']).copy()
df_map = df.dropna(subset=['Latitude', 'Longitude']).copy()


Once the missing-value rules have been applied, the notebook standardizes the remaining raw fields. This part converts IDs, spatial identifiers, booleans, coordinates, and text columns into analysis-friendly types. The goal here is to make the cleaned dataset structurally consistent before deriving any additional time variables.


In [20]:
df['ID'] = pd.to_numeric(df['ID'], errors='coerce').astype('Int32')

df['District'] = pd.to_numeric(df['District'], errors='coerce').astype('Int16')
df['Community Area'] = pd.to_numeric(df['Community Area'], errors='coerce').astype('Int16')
df['Year'] = pd.to_numeric(df['Year'], errors='coerce').astype('Int16')

df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce').astype('float32')
df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce').astype('float32')

df['Arrest'] = df['Arrest'].astype('boolean')
df['Domestic'] = df['Domestic'].astype('boolean')

cat_cols = ['Primary Type', 'Description', 'Location Description']
for col in cat_cols:
    df[col] = df[col].astype('category')

df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')

df_map['ID'] = pd.to_numeric(df_map['ID'], errors='coerce').astype('Int32')
df_map['District'] = pd.to_numeric(df_map['District'], errors='coerce').astype('Int16')
df_map['Community Area'] = pd.to_numeric(df_map['Community Area'], errors='coerce').astype('Int16')
df_map['Year'] = pd.to_numeric(df_map['Year'], errors='coerce').astype('Int16')
df_map['Latitude'] = pd.to_numeric(df_map['Latitude'], errors='coerce').astype('float32')
df_map['Longitude'] = pd.to_numeric(df_map['Longitude'], errors='coerce').astype('float32')
df_map['Arrest'] = df_map['Arrest'].astype('boolean')
df_map['Domestic'] = df_map['Domestic'].astype('boolean')
for col in cat_cols:
    df_map[col] = df_map[col].astype('category')
df_map['Date'] = pd.to_datetime(df_map['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')


The timestamp variable is treated separately because it serves two purposes at once: it is a field whose type must be standardized, and it is also the source of the derived time features used later in EDA and modeling. By converting `Date` first and then extracting `Month`, `Hour`, and `DayOfWeek`, the notebook keeps the logic of time handling in one place.


In [21]:
df['Month'] = df['Date'].dt.month.astype('Int8')
df['Hour'] = df['Date'].dt.hour.astype('Int8')
df['DayOfWeek'] = df['Date'].dt.dayofweek.astype('Int8')

df_map['Month'] = df_map['Date'].dt.month.astype('Int8')
df_map['Hour'] = df_map['Date'].dt.hour.astype('Int8')
df_map['DayOfWeek'] = df_map['Date'].dt.dayofweek.astype('Int8')


## Data Quality Checks

After the row-level cleaning, type standardization, and time-feature derivation are complete, the notebook re-checks missingness, timestamp parsing, duplication, and identifier consistency. This is the point where we verify whether the intended rules resolved the key quality issues and whether any important limitations remain in the final analytical datasets.


In [22]:
print("Missing values in df:")
print(df.isna().sum())

print("\nMissing values in df_map:")
print(df_map.isna().sum())

print("\nUnparsed dates in df:", df['Date'].isna().sum())
print("Unparsed dates in df_map:", df_map['Date'].isna().sum())

Missing values in df:
ID                          0
Date                        0
Primary Type                0
Description                 0
Location Description        0
Arrest                      0
Domestic                    0
District                    0
Community Area              0
Year                        0
Latitude                36953
Longitude               36953
Month                       0
Hour                        0
DayOfWeek                   0
dtype: int64

Missing values in df_map:
ID                      0
Date                    0
Primary Type            0
Description             0
Location Description    0
Arrest                  0
Domestic                0
District                0
Community Area          0
Year                    0
Latitude                0
Longitude               0
Month                   0
Hour                    0
DayOfWeek               0
dtype: int64

Unparsed dates in df: 0
Unparsed dates in df_map: 0


In [23]:
print("Duplicate full rows in df:", df.duplicated().sum())
print("Duplicate IDs in df:", df['ID'].duplicated().sum())

print("\nDuplicate full rows in df_map:", df_map.duplicated().sum())
print("Duplicate IDs in df_map:", df_map['ID'].duplicated().sum())

print("\nYear-Date mismatches in df:", (df['Date'].dt.year != df['Year']).fillna(False).sum())
print("Year-Date mismatches in df_map:", (df_map['Date'].dt.year != df_map['Year']).fillna(False).sum())

Duplicate full rows in df: 0
Duplicate IDs in df: 0

Duplicate full rows in df_map: 0
Duplicate IDs in df_map: 0

Year-Date mismatches in df: 0
Year-Date mismatches in df_map: 0


In [24]:
print(
    "Latitude outside Chicago-like range in df_map:",
    ((df_map['Latitude'] < 41.5) | (df_map['Latitude'] > 42.1)).sum()
)

print(
    "Longitude outside Chicago-like range in df_map:",
    ((df_map['Longitude'] < -88.0) | (df_map['Longitude'] > -87.4)).sum()
)

Latitude outside Chicago-like range in df_map: 4
Longitude outside Chicago-like range in df_map: 4


The post-cleaning checks now verify four things explicitly: the remaining missing-value pattern, whether any timestamps failed to parse, whether IDs remain unique, and whether the map-ready coordinates stay within a plausible Chicago range. Together, these checks provide stronger evidence that the cleaned outputs are reliable for downstream use.

The results support the main cleaning conclusion. Core analytical fields in `df` are complete after cleaning, no duplicate-ID issue is present, `Year` and `Date` remain consistent, and only a very small number of coordinate records require spatial caution.


## Final Outputs and Summary

At this stage, the notebook has produced two cleaned outputs with different purposes:

- `df` is the main analysis dataset. It keeps incidents with complete core predictors and target information, even if geographic coordinates are missing.
- `df_map` is the map-ready dataset. It applies the stricter coordinate requirement so that spatial visualizations are based only on records that can be plotted reliably.

The descriptive summaries below serve as compact evidence that the final datasets are well formed and ready for use.


In [25]:
df.describe()

,ID,Date,District,Community Area,Year,Latitude,Longitude,Month,Hour,DayOfWeek
count,2491668.0,2491668,2491668.0,2491668.0,2491668.0,2.454715e+06,2.454715e+06,2491668.0,2491668.0,2491668.0
mean,12197439.472684,2020-11-30 01:57:23.599163648,11.253122,36.599791,2020.40648,4.184455e+01,-8.766965e+01,6.610433,12.745794,3.012063
min,22245.0,2016-01-01 00:00:00,1.0,1.0,2016.0,3.661945e+01,-9.168657e+01,1.0,0.0,0.0
25%,11308602.75,2018-05-04 20:29:45,6.0,23.0,2018.0,4.176926e+01,-8.771163e+01,4.0,8.0,1.0
50%,12210118.0,2020-10-25 15:00:00,10.0,32.0,2020.0,4.186320e+01,-8.766303e+01,7.0,14.0,3.0
75%,13149307.25,2023-07-13 16:36:15,17.0,53.0,2023.0,4.190715e+01,-8.762745e+01,9.0,18.0,5.0
max,14161190.0,2025-12-31 23:58:00,31.0,77.0,2025.0,4.202267e+01,-8.752453e+01,12.0,23.0,6.0
std,1228494.808984,NaN,7.008925,21.506605,2.922181,1.176823e+00,2.715888e+00,3.363113,6.738574,1.997895


In [26]:
df_map.describe()

,ID,Date,District,Community Area,Year,Latitude,Longitude,Month,Hour,DayOfWeek
count,2454715.0,2454715,2454715.0,2454715.0,2454715.0,2.454715e+06,2.454715e+06,2454715.0,2454715.0,2454715.0
mean,12193362.528357,2020-12-02 09:11:02.366517248,11.245203,36.629777,2020.413219,4.184454e+01,-8.766961e+01,6.603928,12.789305,3.016013
min,22245.0,2016-01-01 00:00:00,1.0,1.0,2016.0,3.661945e+01,-9.168657e+01,1.0,0.0,0.0
25%,11301077.5,2018-05-03 15:59:00,6.0,23.0,2018.0,4.176926e+01,-8.771163e+01,4.0,8.0,1.0
50%,12202343.0,2020-10-25 18:22:00,10.0,32.0,2020.0,4.186320e+01,-8.766303e+01,7.0,14.0,3.0
75%,13150136.5,2023-07-20 10:57:00,17.0,53.0,2023.0,4.190715e+01,-8.762745e+01,9.0,18.0,5.0
max,14159921.0,2025-12-31 23:58:00,31.0,77.0,2025.0,4.202267e+01,-8.752453e+01,12.0,23.0,6.0
std,1232686.426231,NaN,7.008365,21.499781,2.928794,1.176823e+00,2.715888e+00,3.358722,6.728999,1.998696


The cleaned modeling dataset contains **2,491,668** incidents, while the map-ready dataset contains **2,454,715** incidents with valid coordinates. In practical terms, this means the project can move forward with one dataset optimized for statistical analysis and another optimized for spatial display.

Overall, the cleaning results support the rest of the workflow well. The raw data required only limited intervention, the core predictors are complete after cleaning, and the exported datasets provide a stable foundation for both the EDA notebook and the modeling notebook.


## Exported Datasets

The final step writes the cleaned outputs to `df.csv` and `df_map.csv`. These files are treated as the formal inputs for the later project stages.


In [27]:
df.to_csv('/Users/fuyanting/Documents/capstone/data/df.csv', index=False)
df_map.to_csv('/Users/fuyanting/Documents/capstone/data/df_map.csv', index=False)